In [ ]:
# Notebook: Transition from Fourier Series (Discrete Spectrum) to Fourier Transform (Continuous Spectrum)
# Author: Adapted for Springer-style presentation

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, VBox, Output
from IPython.display import display

# ==========================
# Parameters & Signal Definition
# ==========================
A = 1.0
tau = 2.0

# Output container for Jupyter
out = Output()

# Slider to control the period T
T_slider = FloatSlider(
    value=4.0,
    min=4.0,
    max=30.0,
    step=0.5,
    description="Period T",
    style={'description_width': 'initial'}
)

# ==========================
# Plotting Function
# ==========================
def plot_spectrum(change=None):
    with out:
        out.clear_output(wait=True)
        
        T = T_slider.value
        f_0 = 1.0 / T           # Fundamental frequency in Hz (spacing between lines)
        omega_0 = 2 * np.pi / T  # Fundamental frequency in rad/s
        
        # Max frequency limits corresponding to omega_max = 15.0 rad/s
        omega_max = 15.0
        f_max = omega_max / (2 * np.pi)
        
        n_max = int(np.ceil(omega_max / omega_0))
        
        n_vals = np.arange(-n_max, n_max + 1)
        f_vals = n_vals * f_0  # Frequencies in Hz for the stem plot
        
        # 1. Correct physical computation of discrete Fourier Series power spectrum
        args = n_vals * omega_0 * tau / 2
        with np.errstate(divide='ignore', invalid='ignore'):
            sinc_vals = np.where(n_vals == 0, 1.0, np.sin(args) / args)
            
        xn_vals = (A * tau / T) * sinc_vals
        power_spectrum = np.abs(xn_vals)**2
        
        # 2. Continuous envelope function in terms of frequency f
        f_cont = np.linspace(-f_max, f_max, 1000)
        omega_cont = 2 * np.pi * f_cont
        args_cont = omega_cont * tau / 2
        with np.errstate(divide='ignore', invalid='ignore'):
            sinc_cont = np.where(omega_cont == 0, 1.0, np.sin(args_cont) / args_cont)
            
        envelope = (A * tau / T) * sinc_cont
        power_envelope = np.abs(envelope)**2
        
        # Create the figure with extra bottom margin to accommodate the external legend cleanly
        fig, ax = plt.subplots(figsize=(10, 5.5))
        
        # Plot continuous envelope using frequency f
        ax.plot(
            f_cont, power_envelope,
            'r--',
            linewidth=1.5,
            alpha=0.6,
            label=r"Continuous Envelope $\propto |X(j2\pi f)|^2$"
        )
        
        # Plot discrete spectral lines with f0 spacing displayed in the label and using f_vals
        markerline, stemlines, baseline = ax.stem(
            f_vals, power_spectrum,
            linefmt='b-',
            markerfmt='bo',
            basefmt='k-'
        )
        plt.setp(stemlines, linewidth=2, label=f"Discrete Spectrum ($|x_n|^2$), Spacing $f_0$ = {f_0:.3f} Hz")
        plt.setp(markerline, markersize=4)
        
        # Formatting with frequency f on the horizontal axis
        ax.set_xlabel(r"Frequency $f$ (Hz)", fontsize=11)
        ax.set_ylabel(r"Harmonic Power / Magnitude Squared", fontsize=11)
        ax.set_title(
            f"Transition from Fourier Series to Fourier Transform (T = {T:.1f}s, $f_0$ = {f_0:.3f} Hz)",
            fontsize=12
        )
        ax.grid(True, linestyle=":", alpha=0.7)
        ax.set_xlim(-f_max, f_max)
        
        # Dynamic Y-axis scaling based on the peak value for the current T
        peak_val = (A * tau / T)**2
        ax.set_ylim(-0.01 * peak_val, 1.15 * peak_val)
        
        # Place legend below the horizontal axis to prevent it from overlapping with the graph
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, frameon=True)
        fig.subplots_adjust(bottom=0.22)
        
        display(fig)
        plt.close(fig)

# ==========================
# Widget Setup & Initialization
# ==========================
T_slider.unobserve_all()
T_slider.observe(plot_spectrum, names="value")

display(VBox([T_slider, out]))
plot_spectrum()